# Problem 1
Develop an effective BoW representation plus binary Logistic Regression classifier pipeline, aiming to produce the best possible performance on heldout data.

- x_train_df - from x_train.csv, training input data (passages and associated features), each row = 1 passge of text
- y_train_df - from y_train.csv, training labels (correct reading level category for each passage), use to teach model what to predict
- x_test_df - from x_test.csv, test input data (passages we'll be making predictions for)

In [52]:
# loading the data

import pandas as pd

x_train_df = pd.read_csv('data_readinglevel/x_train.csv')
y_train_df = pd.read_csv('data_readinglevel/y_train.csv')
x_test_df = pd.read_csv('data_readinglevel/x_test.csv')

# extract "text" column from training and test data
tr_list_of_text = x_train_df['text'].values.tolist()
te_list_of_text = x_test_df['text'].values.tolist()

# extract correct training labels ("coarse label")
y_train = (y_train_df['Coarse Label'] == 'Key Stage 4-5').astype(int).values # Key Stage 4-5 = positive class

print("Training examples:", len(tr_list_of_text))
print("Training labels:", len(y_train))
print("Test examples:", len(te_list_of_text))

# how many training examples belong in each class (are the two classes balanced?)
print(y_train_df['Coarse Label'].value_counts())

Training examples: 5557
Training labels: 5557
Test examples: 1197
Coarse Label
Key Stage 4-5    3048
Key Stage 2-3    2509
Name: count, dtype: int64


In [29]:
# build a baseline model 
# this is the simplest version of the required model (BoW + logistic regression)

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# custom_pattern = r"(?u)\b\w+[\'\w]*\b"

# create reusable pipeline function
def create_model(min_df=1, binary=False, stop_words=None, C=1.0):
  return Pipeline([
    # ("vectorizer", CountVectorizer(min_df=min_df, binary=binary, stop_words=stop_words, token_pattern=custom_pattern)),
    ("vectorizer", CountVectorizer(min_df=min_df, binary=binary, stop_words=stop_words)),
    ("classifier", LogisticRegression(C=C, max_iter=1000))
  ])

# logistic regression learns how numerical features from CountVectorizer relate to the labels
base_model = create_model()

In [27]:
# evaluate the baseline model with cross-validation
# benchmark to compare against preprocessing experiments

import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# create a cross-validation function
def evaluate_model(model, x, y, cv):
  results = cross_validate(model, x, y, cv=cv, scoring="roc_auc", return_train_score=True)

  print(f"Mean training AUROC: {results['train_score'].mean():.4f}")
  print(f"Mean validation AUROC: {results['test_score'].mean():.4f}")
  print(f"Validation AUROC std: {results['test_score'].std():.4f}")
  print("\n")

  return results

base_results = evaluate_model(base_model, tr_list_of_text, y_train, cv)

Mean training AUROC: 1.0000
Mean validation AUROC: 0.8115
Validation AUROC std: 0.0090




Baseline model fits the training data almost perfectly, but is worse on unseen validation data

In [47]:
# compare multiple configurations

def compare_configs(configs, X, y, cv):
  results = []
  
  for name, config in configs.items():
    model = create_model(**config)
    scores = cross_validate(model, X, y, cv=cv, scoring="roc_auc", return_train_score=True)

    results.append({
      "name": name,
      **config,
      "train_mean": scores["train_score"].mean(),
      "train_std": scores["train_score"].std(),
      "val_mean": scores["test_score"].mean(),
      "val_std": scores["test_score"].std(),
      "fold_scores": scores["test_score"]
    })

  return pd.DataFrame(results)

In [50]:
bow_configs = {
  "baseline": {},
  "min_df2": {"min_df": 2},
  "min_df5": {"min_df": 5},
  "binary": {"binary": True},
  "stopwords": {"stop_words": "english"}
}

bow_results = compare_configs(bow_configs, tr_list_of_text, y_train, cv)
print(bow_results[
  ["name", "train_mean", "val_mean", "val_std"]
])

        name  train_mean  val_mean   val_std
0   baseline    1.000000  0.818555  0.009549
1    min_df2    0.999999  0.813947  0.010841
2    min_df5    0.999922  0.797453  0.008445
3     binary    0.999998  0.807527  0.009383
4  stopwords    1.000000  0.822292  0.010562


In [35]:
# fit the final model with chosen BoW configuration

final_model = create_model(stop_words="english")
final_model.fit(tr_list_of_text, y_train)

vocab_size = len(final_model.named_steps["vectorizer"].vocabulary_)
print("Final vocab size:", vocab_size)

Final vocab size: 26561


___

In [ ]:
# investigating which cv method to use
# does the same author appear in different folds?
# model is supposed to learn patterns in text that help it predict reading level